# Stage 2A：Math-LiC 题目渐进分片与确定性对齐审计（Colab A100）

本 Notebook 在 **Google Colab A100 (40GB)** 上使用 `vLLM` 运行 `Qwen/Qwen3-32B-AWQ`（关闭长思考模式 `enable_thinking=False`，`temperature=0.0`，开启 JSON Schema 结构化约束），完成 **Dev 20 + Test 100 + Reserve 120 = 240 题** 的渐进分片与确定性对齐校验。

### 运行前准备
1. 菜单栏 **代码执行程序 -> 更改运行时类型 -> 选择 `A100 GPU`**。
2. 点击左侧文件夹图标，将本地压缩包 **`colab_stage2_sharding.zip`** 上传到 `/content/` 目录。
3. 从上往下依次运行全部单元格即可。

In [ ]:
#@title 1. 检查 A100 GPU、解压工作包并安装 vLLM
!nvidia-smi
import zipfile
from pathlib import Path

WORK_DIR = Path('/content/colab_stage2_sharding')
ZIP_PATH = Path('/content/colab_stage2_sharding.zip')

if ZIP_PATH.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(WORK_DIR)
    print('已解压到:', WORK_DIR)
elif not WORK_DIR.exists():
    raise FileNotFoundError('请先将 colab_stage2_sharding.zip 上传到 Colab 左侧 /content/ 目录！')

%cd /content/colab_stage2_sharding
!ls -lh
!python audit_alignment.py --self-test
!pip install -q -U vllm

In [ ]:
#@title 2. 可选（推荐）：挂载 Google Drive 自动备份输出，防止 Colab 断连丢结果
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cscwd_v2_stage2a

In [ ]:
#@title 3. 先在 Dev 20 题上试跑分片 + 自动修复 + 确定性审计（约 1 分钟）
# 模型默认使用 Qwen/Qwen3-32B-AWQ（可通过 --model 换为 Qwen/Qwen2.5-72B-Instruct-AWQ）
!python run_colab_a100_sharding.py \
  --model Qwen/Qwen3-32B-AWQ \
  --input stage2_input_candidates.jsonl \
  --prompt prompt_math_lic_v2_6.txt \
  --status-filter SELECTED_DEV \
  --output dev20_sharded.jsonl \
  --summary dev20_summary.json

!cp -f dev20_sharded.jsonl dev20_summary.json /content/drive/MyDrive/cscwd_v2_stage2a/ 2>/dev/null || true

In [ ]:
#@title 4. 抽查 Dev 20 题的分片详情与对齐审计结果（不打印 Test，保持测试集盲态）
import json

with open('dev20_sharded.jsonl', 'r', encoding='utf-8') as fh:
    dev_rows = [json.loads(line) for line in fh if line.strip()]

for r in dev_rows[:5]:
    print('====================================')
    print(f"{r['item_id']} ({r['macro_subfield']} / {r['provisional_mechanism']}) | attempts={r['attempts']}")
    print('原题:', r['question'])
    print('审计结果:', r['audit']['status'], 'passed=', r['audit']['alignment_passed'])
    if r['audit'].get('errors'):
        print('错误:', r['audit']['errors'])
    p = r['parsed_output']
    if p and p.get('status') == 'CANDIDATE':
        print('  S1:', p['initial_shard'], '| segments:', p['initial_segments'])
        for s in p['shards']:
            print(f"  {s['shard_id']}:", s['shard'], '| segments:', s['segments'])
    elif p:
        print('  UNSHARDABLE:', p.get('reason_code'), p.get('reason'))

In [ ]:
#@title 5. 全量运行 240 题（Dev 20 + Test 100 + 每桶前 15 候补 120 题）并打包下载
!python run_colab_a100_sharding.py \
  --model Qwen/Qwen3-32B-AWQ \
  --input stage2_input_candidates.jsonl \
  --prompt prompt_math_lic_v2_6.txt \
  --status-filter ALL \
  --output stage2a_sharded_candidates.jsonl \
  --summary stage2a_audit_summary.json

!cp -f stage2a_sharded_candidates.jsonl stage2a_audit_summary.json /content/drive/MyDrive/cscwd_v2_stage2a/ 2>/dev/null || true

import zipfile
with zipfile.ZipFile('/content/stage2a_outputs.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write('stage2a_sharded_candidates.jsonl')
    zf.write('stage2a_audit_summary.json')

from google.colab import files
files.download('/content/stage2a_outputs.zip')